In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [4]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your operating system.
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To test that the local server is running, you can use:
```bash
curl http://localhost:11434
```

If you get a connection refused error while using Ollama in the homework, restart the server with:
```bash
!nohup ollama serve > nohup.out 2>&1 &
```


In [5]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t see any FAQ entry about running **Ollama** locally.

The closest related FAQ says you can run the **course** locally instead of Codespaces if you’re comfortable setting up Python, `uv`, Jupyter, Docker, and other needed tools, and that you should document your setup to keep it reproducible.

If you meant something specific about Ollama setup, it isn’t covered in the provided context.


First, let's see what the LLM does without any tools. We ask it a course-specific question and look at the answer.

In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Probably yes — if the course is still open, you can usually join after discovering it.\n\nTo confirm, I’d need a bit more detail:\n- the course name/link\n- whether it’s self-paced or has a live start date\n- where you found it\n\nIf you want, I can help you figure out:\n1. whether enrollment is still open, or  \n2. how to contact the instructor/organizer to ask for late enrollment.'

First we define a top-level search function that queries the index directly. The model will reference it by this name. We keep the Python function and the tool name aligned so the dispatch is easier later.

In [5]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [9]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [10]:
len(response.output)

1

In [11]:
call = response.output[0]

In [12]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered new student enrollment can I join"}', call_id='call_NiqgQ7dgX1cUAIcAOEmcsoaT', name='search', type='function_call', id='fc_02a96200b05b1859006a36f1cd01dc8199990956835b345089', namespace=None, status='completed')

In [6]:
import json

args = json.loads(call.arguments)
args

NameError: name 'call' is not defined

In [14]:
call.name

'search'

In [15]:
results = search(**args)

In [16]:
result_json = json.dumps(results, indent=2)

In [17]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [18]:
messages.append(call)

In [19]:
messages.append(function_call_output)

In [20]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered new student enrollment can I join"}', call_id='call_NiqgQ7dgX1cUAIcAOEmcsoaT', name='search', type='function_call', id='fc_02a96200b05b1859006a36f1cd01dc8199990956835b345089', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_NiqgQ7dgX1cUAIcAOEmcsoaT',
  'output': '[\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.",\n    "doc_id": "74eb249bbf"\n  },\n  {\n    "course": "llm-zoomcamp",\n    "section": "Module 2: Vector Search",\n    "question": "Do I need a new GitHub repo for Module 2, or just a new codespace?",\n    "answer": "Just a new codespace. A code

In [21]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [22]:
print(response.output_text)

Yes — you can still join. You can start learning and submitting homework while the submission form is open.

If you want a certificate, make sure you submit your project while submissions are still being accepted.


In [23]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(783, 44)

In [24]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


In [25]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

## Agentic loop

In [7]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [8]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [10]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes, you can still join the course.

You can start learning and submitting homework even if you just discovered it. Registration isn’t required to begin, and it’s mainly used to gauge interest. If you want a certificate, make sure you submit your project while submissions are still open.

If you’d like, I can also help you figure out how to get started with the course workflow. Any other areas you want to explore?


In [11]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [12]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run Ollama local install run locally"}
function_call: search {"query":"Ollama local setup run model locally course FAQ"}
iteration #2...
ASSISTANT:
If you mean **Ollama** locally, the basic steps are:

1. **Install Ollama**
   - macOS: download the `.pkg` from https://ollama.com/download
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Run a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model and start a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from the Ollama server.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['messag

'If you mean **Ollama** locally, the basic steps are:\n\n1. **Install Ollama**\n   - macOS: download the `.pkg` from https://ollama.com/download\n   - Windows: download the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Run a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and start a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from the Ollama server.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, you may need to start the server manually:\n\n```bash\nollama serve\n```\n\nIf you want, I can al

In [13]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered course can I still join"}
function_call: search {"query":"enrollment late join course discovered course"}
function_call: search {"query":"can I still join the course after it started"}
iteration #2...
ASSISTANT:
Yes, you can still join.

You can start anytime since the videos and materials are available. If you want a certificate, make sure you submit your project while submissions are still open. Also, if the course is still running live, you can participate in the homework and capstone process; registration itself is not required to start learning.

If you want, I can also help with how to get started week by week or explain the certificate requirements.


'Yes, you can still join.\n\nYou can start anytime since the videos and materials are available. If you want a certificate, make sure you submit your project while submissions are still open. Also, if the course is still running live, you can participate in the homework and capstone process; registration itself is not required to start learning.\n\nIf you want, I can also help with how to get started week by week or explain the certificate requirements.'

In [14]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join the course enrollment discovered the course can I join late FAQ"}
iteration #2...
function_call: search {"query":"self-paced mode certificate project submissions while accepting submissions join now live cohort FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

A couple of important notes:
- You can start learning and working through the materials even if you discovered it late.
- If you want a certificate, you need to submit your project while submissions are still open.

If you want, I can also explain how to get started with the course workflow. Are there other areas you want to explore?


'Yes — you can still join the course.\n\nA couple of important notes:\n- You can start learning and working through the materials even if you discovered it late.\n- If you want a certificate, you need to submit your project while submissions are still open.\n\nIf you want, I can also explain how to get started with the course workflow. Are there other areas you want to explore?'

In [15]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit Queen's Gambit chess opening"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening definition"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the c-pawn to try to gain control of the center, especially the **e4** square. It’s one of the oldest and most popular openings in chess.

A quick note: the search results I found didn’t contain a course FAQ entry about chess, so I’m answering from general knowledge here.

If you want, I can also explain:
- the idea behind the Queen’s Gambit,
- the difference between **Queen’s Gambit Accepted** and **Declined**,
- or show a few common opening moves.


'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the c-pawn to try to gain control of the center, especially the **e4** square. It’s one of the oldest and most popular openings in chess.\n\nA quick note: the search results I found didn’t contain a course FAQ entry about chess, so I’m answering from general knowledge here.\n\nIf you want, I can also explain:\n- the idea behind the Queen’s Gambit,\n- the difference between **Queen’s Gambit Accepted** and **Declined**,\n- or show a few common opening moves.'

In [13]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()


In [ ]:
agent_loop(instructions, "what's queen gambit?")

## ToyAIKit

In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [8]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [9]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [10]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [11]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [14]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [15]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [16]:
result.cost

CostInfo(input_cost=Decimal('0.0027345'), output_cost=Decimal('0.0013995'), total_cost=Decimal('0.0041340'))

In [17]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Ollama local run install run locally FAQ"}', call_id='call_sW4

In [18]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [19]:
runner.run()

-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='Is llama3 better?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"llama3 better"}', call_id='call_xNzJ404hMu4VSeoFUm

In [ ]:
s